# NSE Symbol Lineage Walkthrough

NSE tickers change over time — companies rename themselves, merge, demerge, or get delisted.
If your analytics pipeline stores data by ticker symbol, a rename silently breaks your history.

This notebook shows:
- What lineage events look like in TickerTruth data
- How ISIN remains stable across renames and mergers
- How to reconstruct the full history for a security regardless of symbol changes

**Data source:** `sample_data/` — a curated subset of the TickerTruth NSE reference dataset.

In [ ]:
import pandas as pd
from pathlib import Path

# Resolve sample_data/ whether running from project root or notebooks/
_cwd = Path.cwd()
SAMPLE_DATA = (
    _cwd / "sample_data"
    if (_cwd / "sample_data").exists()
    else _cwd.parent / "sample_data"
)

security = pd.read_parquet(SAMPLE_DATA / "security_master.parquet")
lineage = pd.read_parquet(SAMPLE_DATA / "symbol_lineage.parquet")

print(f"Securities: {len(security)} rows")
print(f"Lineage events: {len(lineage)} rows")

## Security master — current state

The security master records the *current* symbol and ISIN for each listed security.
ISINs are assigned by depositories (NSDL/CDSL) and survive symbol changes.

In [ ]:
security[["symbol", "isin", "company_name", "listing_date", "trading_status"]]

## Lineage events — what changed and when

Each row records a single symbol change event with:
- `old_symbol` / `new_symbol` — the before and after tickers
- `effective_date` — date the change took effect on NSE
- `event_type` — RENAME, MERGER, DEMERGER, DELISTING, RELISTING
- `reason` — source annotation

In [ ]:
lineage[["old_symbol", "new_symbol", "effective_date", "event_type", "reason"]]

## Tracing a security through a merger

ZENITHEQUIP merged into ICIL in June 2022. Anyone holding data keyed by the old ticker
would lose continuity from that date forward. TickerTruth maps the merger so you can
stitch historical ZENITHEQUIP data to post-merger ICIL data correctly.

In [ ]:
merger = lineage[lineage["event_type"] == "MERGER"].iloc[0]
print(f"Old ticker:      {merger['old_symbol']}")
print(f"New ticker:      {merger['new_symbol']}")
print(f"Effective date:  {merger['effective_date'].date()}")
print(f"Reason:          {merger['reason']}")
print()
print(
    "Any price series stored under ZENITHEQUIP must be continued under ICIL after this date."
)
print("Without lineage data, the two series look like two unrelated securities.")

## ISIN as the stable anchor

When a ticker changes, the ISIN stays the same (unless it is a genuine new security).
Using ISIN as the join key in your pipeline eliminates symbol-change breakage entirely.

In [ ]:
# Simulate a query: given old_symbol AEGISLOG, find the current symbol and ISIN
old_sym = "AEGISLOG"
event = lineage[lineage["old_symbol"] == old_sym].iloc[0]
current_sym = event["new_symbol"]

current = security[security["symbol"] == current_sym]
print(f"Old symbol:      {old_sym}")
print(f"New symbol:      {current_sym}  (renamed {event['effective_date'].date()})")
if not current.empty:
    print(f"ISIN:            {current.iloc[0]['isin']}  ← unchanged through rename")
    print(f"Company:         {current.iloc[0]['company_name']}")
    print(f"Status:          {current.iloc[0]['trading_status']}")

## Summary of event types in this dataset

In [ ]:
summary = lineage.groupby("event_type").agg(
    count=("old_symbol", "count"),
    earliest=("effective_date", "min"),
    latest=("effective_date", "max"),
)
summary["earliest"] = summary["earliest"].dt.date
summary["latest"] = summary["latest"].dt.date
print(summary.to_string())

---
## Key takeaway

| Problem | Without TickerTruth | With TickerTruth |
|---|---|---|
| Ticker renamed | History breaks at rename date | Lineage event maps old → new |
| Company merged | Two disconnected series | Merger event links predecessor |
| Symbol reused by new company | Silent data contamination | Lineage flags the delisting + new listing |

For any pipeline joining price data to fundamentals or corporate actions by ticker,
lineage events are the prerequisite — not an optional enrichment.